In [1]:
import random

In [2]:
lang_src = [0, 1, 2, 3, 4, 5]
lang_tgt_base = [6, 7, 8, 9, 10, 11]
lang_tgt_source = [12, 13, 14, 15, 16, 17]

lang_letters = ['urd', 'loe', 'clight', 'nitty', 'keam', 'frue',
                'meud', 'zoin', 'haw', 'digh', 'pleam', 'stowl',
                'bell', 'traip', 'verk', 'yart', 'arn', 'jerth']

In [3]:
def create_prompt_pair(num_shots=20):
    phrases_src = []
    for shot_i in range(num_shots):
        phrase_src = [random.randint(0, 5), random.randint(0, 5)]
        while phrase_src in phrases_src:
            phrase_src = [random.randint(0, 5), random.randint(0, 5)]
        phrases_src.append(phrase_src)
    
    prompt_base = ""
    prompt_source = ""
    for phrase in phrases_src:
        prompt_base += f"\"{lang_letters[lang_src[phrase[0]]]} {lang_letters[lang_src[phrase[1]]]}\" - \"{lang_letters[lang_tgt_base[phrase[1]]]} {lang_letters[lang_tgt_base[phrase[0]]]}\"\n"
        prompt_source += f"\"{lang_letters[lang_src[phrase[0]]]} {lang_letters[lang_src[phrase[1]]]}\" - \"{lang_letters[lang_tgt_source[phrase[0]]]} {lang_letters[lang_tgt_source[phrase[1]]]}\"\n"
    
    last_prompt_base = [random.randint(0, 5), random.randint(0, 5)]
    last_prompt_source = [random.randint(0, 5), random.randint(0, 5)]

    prompt_base += f"\"{lang_letters[lang_src[last_prompt_base[0]]]} {lang_letters[lang_src[last_prompt_base[1]]]}\" - \""
    prompt_source += f"\"{lang_letters[lang_src[last_prompt_source[0]]]} {lang_letters[lang_src[last_prompt_source[1]]]}\" - \""

    answers = {
        "baseorder_baseconcept_baselang": lang_letters[lang_tgt_base[last_prompt_base[1]]],
        "sourceorder_baseconcept_baselang": lang_letters[lang_tgt_base[last_prompt_base[0]]],
        "baseorder_baseconcept_sourcelang": lang_letters[lang_tgt_source[last_prompt_base[1]]],
        "sourceorder_baseconcept_sourcelang": lang_letters[lang_tgt_source[last_prompt_base[0]]],
        
        "baseorder_sourceconcept_baselang": lang_letters[lang_tgt_base[last_prompt_source[1]]],
        "sourceorder_sourceconcept_baselang": lang_letters[lang_tgt_base[last_prompt_source[0]]],
        "baseorder_sourceconcept_sourcelang": lang_letters[lang_tgt_source[last_prompt_source[1]]],
        "sourceorder_sourceconcept_sourcelang": lang_letters[lang_tgt_source[last_prompt_source[0]]],
    }

    return (prompt_base, prompt_source), answers

In [4]:
(prompt_base, prompt_source), ans = create_prompt_pair()

In [5]:
print(prompt_base)

"loe keam" - "pleam zoin"
"urd nitty" - "digh meud"
"frue nitty" - "digh stowl"
"clight clight" - "haw haw"
"keam urd" - "meud pleam"
"loe clight" - "haw zoin"
"keam nitty" - "digh pleam"
"clight keam" - "pleam haw"
"loe nitty" - "digh zoin"
"frue loe" - "zoin stowl"
"nitty keam" - "pleam digh"
"loe frue" - "stowl zoin"
"nitty urd" - "meud digh"
"urd urd" - "meud meud"
"nitty nitty" - "digh digh"
"urd frue" - "stowl meud"
"keam loe" - "zoin pleam"
"frue clight" - "haw stowl"
"clight nitty" - "digh haw"
"frue keam" - "pleam stowl"
"frue keam" - "


In [6]:
print(prompt_source)

"loe keam" - "traip arn"
"urd nitty" - "bell yart"
"frue nitty" - "jerth yart"
"clight clight" - "verk verk"
"keam urd" - "arn bell"
"loe clight" - "traip verk"
"keam nitty" - "arn yart"
"clight keam" - "verk arn"
"loe nitty" - "traip yart"
"frue loe" - "jerth traip"
"nitty keam" - "yart arn"
"loe frue" - "traip jerth"
"nitty urd" - "yart bell"
"urd urd" - "bell bell"
"nitty nitty" - "yart yart"
"urd frue" - "bell jerth"
"keam loe" - "arn traip"
"frue clight" - "jerth verk"
"clight nitty" - "verk yart"
"frue keam" - "jerth arn"
"frue loe" - "


In [7]:
ans

{'baseorder_baseconcept_baselang': 'pleam',
 'sourceorder_baseconcept_baselang': 'stowl',
 'baseorder_baseconcept_sourcelang': 'arn',
 'sourceorder_baseconcept_sourcelang': 'jerth',
 'baseorder_sourceconcept_baselang': 'zoin',
 'sourceorder_sourceconcept_baselang': 'stowl',
 'baseorder_sourceconcept_sourcelang': 'traip',
 'sourceorder_sourceconcept_sourcelang': 'jerth'}

In [8]:
model_id = "ai-forever/mGPT"
cache_dir = "/scratch/msonkin/word-order-thesis/cache/"

In [9]:
import torch
import pandas as pd
from tqdm import tqdm
import plotly.express as px
from typing import List, Dict, Any
from transformers import AutoModelForCausalLM, AutoTokenizer

import pyvene as pv
from pyvene import embed_to_distrib, top_vals, format_token

from create_datasets.parallel_dataset import ParallelDataset

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
sm = torch.nn.Softmax(dim=2)

nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.


In [10]:
model = AutoModelForCausalLM.from_pretrained(model_id, cache_dir=cache_dir).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)

In [11]:
def intervention_config(model_type, intervention_type, unit, layer):
    """
    Parameters
    __________

    model_type: model type
    intervention_type: component for RepresentationConfig, e.g. head_attention_value_output
    unit: string to define the component type, e.g. "h" (head), "pos" (position), "h.pos" (head within position)
    later: layer id
    """
    # Set up the config to intervene
    config = pv.IntervenableConfig(
        model_type=model_type,
        representations=[
            pv.RepresentationConfig(
                layer,  # layer
                intervention_type,  # intervention type
                unit,  # intervention unit is now [pos] within [h]
                1,  # max number of unit
            ),
        ],
        intervention_types=pv.VanillaIntervention,
    )
    return config

In [12]:
def intervention_data(
        base: torch.Tensor,
        source: torch.Tensor,
        base_pos: int,
        source_pos: int,
        tokentype2token: Dict[str, str],
        component_type: str,
        head_i: int = None,
        data: List[Dict[str, Any]] = None
    ) -> List[Dict[str, Any]]:
    """
    Collect intervention data for a given model component (block output or head attention value output).

    Parameters
    ----------
    base : torch.Tensor
        The tokenized base prompt tensor.
    source : torch.Tensor
        The tokenized source prompt tensor.
    base_pos : int
        The position of the last token in the base prompt.
    source_pos : int
        The position of the last token in the source prompt.
    tokentype2token : Dict[str, str]
        A dictionary mapping token types (e.g., 'noun-base', 'adj-base') to their corresponding token strings.
    component_type : str
        The type of model component to intervene on. Must be either 'block_output' or 'head_attention_value_output'.
    head_i : int, optional
        The index of the head to intervene on (if applicable). Only used for head-level interventions.
    data : List[Dict[str, Any]], optional
        A list to append the collected data to. If None, a new list will be created.

    Returns
    -------
    List[Dict[str, Any]]
        A list of dictionaries containing the intervention data, with keys:
        - "token_type": The type of token (e.g., 'noun-base', 'adj-base').
        - "token": The token string.
        - "prob": The probability of the token after intervention.
        - "layer": The layer index.
        - "head_i": The head index (if applicable).
        - "pos": The position index.
        - "type": The component type (e.g., 'block_output').

    Raises
    ------
    NotImplementedError
        If the component_type is not 'block_output' or 'head_attention_value_output'.
    """
    if data is None:
        data = []
    for layer_i in range(model.config.n_layer):
        if component_type == "block_output":
            unit = "pos"
        elif component_type == "head_attention_value_output":
            unit = "h.pos"
        else:
            raise NotImplementedError("Unsupported component type: {component_type}")
        config = intervention_config(
            type(model), component_type, unit, layer_i
        )
        intervenable = pv.IntervenableModel(config, model)
        if head_i is not None:
            unit_locations = {
                "sources->base": (
                    [[[[head_i]], [[source_pos]]]],  # intervene w/ target_head's pos_i
                    [[[[head_i]], [[base_pos]]]]
                ),
            }
        else:
            unit_locations = {"sources->base": (source_pos, base_pos)}
        # print(unit_locations)
        # print(base)
        # print(source)
        _, counterfactual_outputs = intervenable(
            base,
            source,
            unit_locations,
        )
        with torch.inference_mode():
            distrib = sm(counterfactual_outputs.logits)
        # print(f"\nTOP VALUES AT LAYER #{layer_i} AT POSITION {base_pos}:")
        # top_vals(tokenizer, distrib[0][base_pos], 5)
        for token_type, token in tokentype2token.items():
            data.append(
                {
                    "token_type": token_type,
                    "token": token,
                    "prob": float(distrib[0][base_pos][tokenizer.encode(token)[0],]),
                    "layer": layer_i,
                    "head_i": head_i,
                    "pos": base_pos,
                    "type": component_type,
                }
            )
    return data

In [13]:
data = []

prompt_pairs = [create_prompt_pair() for _ in range(50)]

for (prompt_base, prompt_source), ans in prompt_pairs:
    # tokenize prompts
    prompt_base = tokenizer(prompt_base, return_tensors="pt").to(device)
    prompt_source = tokenizer(prompt_source, return_tensors="pt").to(device)
    # last token index
    pos_base = prompt_base.input_ids.size(1) - 1
    pos_source = prompt_source.input_ids.size(1) - 1
    # token type to token dict
    tokentype2token = ans
    # print(tokentype2token)

    data = intervention_data(
        prompt_base, 
        prompt_source, 
        pos_base,
        pos_source,
        tokentype2token, 
        "block_output", 
        data=data
    )
df = pd.DataFrame(data)

In [14]:
# Create a line plot for token probabilities over layers
fig = px.line(
    df.groupby(['token_type', 'layer']).mean().reset_index(),
    x="layer",
    y="prob",
    color="token_type",
    title=f"Probabilities after Block Intervention)",
    labels={"layer": "Layer", "prob": "Probability", "token_type": "Token"},
    # category_orders={"layer": [str(i) for i in range(model.config.n_layer)]},
)

# Show the plot
fig.show()

/var/lib/condor/execute/dir_550366/ipykernel_106/1309690226.py:3: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df.groupby(['token_type', 'layer']).mean().reset_index(),


In [ ]:
data = []
source_df_list = list(source_df.iterrows())

for row_i, row in base_dataset.df.iterrows():
    # tokenize prompts
    prompt_base = tokenizer(base_prompts[row_i], return_tensors="pt").to(device)
    prompt_source = tokenizer(source_prompts[row_i], return_tensors="pt").to(device)
    # last token index
    pos_base = prompt_base.input_ids.size(1) - 1
    pos_source = prompt_source.input_ids.size(1) - 1
    # token type to token dict
    tokentype2token = {
        f"noun-base-{tgt_lang_base}": row[f'{noun_base_prefix}{tgt_lang_base}'],
        f"adj-base-{tgt_lang_base}": row[f'{adj_base_prefix}{tgt_lang_base}'],
        f"noun-source-{tgt_lang_base}": source_df_list[row_i][1][f'{noun_source_prefix}{tgt_lang_base}'],
        f"adj-source-{tgt_lang_base}": source_df_list[row_i][1][f'{adj_source_prefix}{tgt_lang_base}'],

        f"noun-base-{tgt_lang_source}": row[f'{noun_base_prefix}{tgt_lang_source}'],
        f"adj-base-{tgt_lang_source}": row[f'{adj_base_prefix}{tgt_lang_source}'],
        f"noun-source-{tgt_lang_source}": source_df_list[row_i][1][f'{noun_source_prefix}{tgt_lang_source}'],
        f"adj-source-{tgt_lang_source}": source_df_list[row_i][1][f'{adj_source_prefix}{tgt_lang_source}']
    }

    for head_i in range(model.config.n_head):
        data = intervention_data(
            prompt_base, 
            prompt_source, 
            pos_base,
            pos_source,
            tokentype2token, 
            "head_attention_value_output", 
            head_i=head_i,
            data=data,
        )
df = pd.DataFrame(data)